# 01 — API Collection (jolpica-f1)

**Project:** F1 Track Personality 2020–2025  
**Authors:** Matteo Asscher & Aksel  
**Goal:** Pull all race results from the jolpica-f1 API for the 2020–2025 F1 seasons and save the raw JSON to `data/raw/` as a checkpoint.

**API used:** jolpica-f1 (Ergast-compatible)  
**Base URL:** `https://api.jolpi.ca/ergast/f1/`  
**Auth:** none required  
**Rate limit:** 200 requests/hour  
**Endpoint:** `/{season}/results/`

**Status (last run):** Pulled all 2020–2025 race results. **2,618 unique driver-result rows** across **138 races**, saved to `data/raw/f1_results_2020_2025.json` and `f1_results_2020_2025.csv`. Zero missing values across all 19 columns. Two known shape notes: 2020 season is shorter (17 races) due to COVID; 2025 is still in progress at time of pull._

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Setup OK")
print(f"pandas:   {pd.__version__}")
print(f"requests: {requests.__version__}")

Setup OK
pandas:   3.0.3
requests: 2.34.2


In [2]:
# Test the API with a single race: 2024 Bahrain Grand Prix (round 1)
url = "https://api.jolpi.ca/ergast/f1/2024/1/results/"
r = requests.get(url, timeout=10)

print("Status code:", r.status_code)

data = r.json()
race = data["MRData"]["RaceTable"]["Races"][0]

print("Race:    ", race["raceName"])
print("Circuit: ", race["Circuit"]["circuitName"])
print("Country: ", race["Circuit"]["Location"]["country"])
print("Date:    ", race["date"])
print("Winner:  ", race["Results"][0]["Driver"]["familyName"], 
      "(" + race["Results"][0]["Constructor"]["name"] + ")")
print(f"\nNumber of finishers in this race: {len(race['Results'])}")

Status code: 200
Race:     Bahrain Grand Prix
Circuit:  Bahrain International Circuit
Country:  Bahrain
Date:     2024-03-02
Winner:   Verstappen (Red Bull)

Number of finishers in this race: 20


In [3]:
def get_json(url, params=None, max_retries=3, timeout=10):
    """
    Robust GET request that returns parsed JSON.
    
    Handles three failure modes:
    - 429 (rate limited): waits Retry-After seconds, then retries
    - 5xx (server error): waits with exponential backoff, then retries
    - 4xx (our bug): raises immediately
    """
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=timeout)
        
        if r.status_code == 200:
            return r.json()
        
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 2 ** attempt))
            print(f"  Rate limited (429). Waiting {wait}s before retry...")
            time.sleep(wait)
            continue
        
        if 500 <= r.status_code < 600:
            wait = 2 ** attempt
            print(f"  Server error ({r.status_code}). Waiting {wait}s before retry...")
            time.sleep(wait)
            continue
        
        # 4xx other than 429 = our fault, fail loudly
        r.raise_for_status()
    
    raise RuntimeError(f"Failed after {max_retries} attempts: {url}")


# Quick smoke test on the same Bahrain URL
test = get_json("https://api.jolpi.ca/ergast/f1/2024/1/results/")
print("Helper works. Race name:", test["MRData"]["RaceTable"]["Races"][0]["raceName"])

Helper works. Race name: Bahrain Grand Prix


In [4]:
BASE_URL = "https://api.jolpi.ca/ergast/f1/{season}/results/"
SEASONS = [2020, 2021, 2022, 2023, 2024, 2025]
PAGE_SIZE = 100  # max allowed by jolpica

all_races = []  # one entry per race, each containing its Results list

for season in SEASONS:
    print(f"\nSeason {season}...")
    offset = 0
    season_race_count = 0
    
    while True:
        url = BASE_URL.format(season=season)
        params = {"limit": PAGE_SIZE, "offset": offset}
        
        data = get_json(url, params=params)
        races = data["MRData"]["RaceTable"]["Races"]
        total = int(data["MRData"]["total"])
        
        all_races.extend(races)
        season_race_count += len(races)
        
        offset += PAGE_SIZE
        if offset >= total:
            break
        
        time.sleep(0.3)  # polite delay between paginated calls
    
    print(f"  collected {season_race_count} result rows across the season")
    time.sleep(0.5)  # polite delay between seasons

print(f"\n✅ Total race entries collected: {len(all_races)}")


Season 2020...
  collected 17 result rows across the season

Season 2021...
  collected 22 result rows across the season

Season 2022...
  collected 22 result rows across the season

Season 2023...
  collected 22 result rows across the season

Season 2024...
  collected 28 result rows across the season

Season 2025...
  collected 27 result rows across the season

✅ Total race entries collected: 138


In [5]:
# Each "race object" contains a list of driver Results inside.
# Flatten so we have one row per (race, driver) combination.

records = []
for race in all_races:
    season     = int(race["season"])
    round_num  = int(race["round"])
    race_name  = race["raceName"]
    circuit    = race["Circuit"]
    
    for result in race.get("Results", []):
        records.append({
            "season":           season,
            "round":            round_num,
            "race_name":        race_name,
            "circuit_id":       circuit["circuitId"],
            "circuit_name":     circuit["circuitName"],
            "country":          circuit["Location"]["country"],
            "date":             race["date"],
            "position":         int(result["position"]),
            "position_text":    result["positionText"],
            "points":           float(result["points"]),
            "grid":             int(result["grid"]),
            "laps":             int(result["laps"]),
            "status":           result["status"],
            "driver_id":        result["Driver"]["driverId"],
            "driver_code":      result["Driver"].get("code", ""),
            "driver_first":     result["Driver"]["givenName"],
            "driver_last":      result["Driver"]["familyName"],
            "constructor_id":   result["Constructor"]["constructorId"],
            "constructor_name": result["Constructor"]["name"],
        })

df = pd.DataFrame(records)

# A race spanning a pagination boundary can appear twice in all_races,
# producing duplicate (season, round, driver_id) rows. Dedupe them.
before = len(df)
df = df.drop_duplicates(subset=["season", "round", "driver_id"]).reset_index(drop=True)
after = len(df)

print(f"Built {before} rows from JSON")
print(f"Removed {before - after} duplicates (from page-split boundaries)")
print(f"Final DataFrame: {after} unique driver-result rows")

Built 2618 rows from JSON
Removed 0 duplicates (from page-split boundaries)
Final DataFrame: 2618 unique driver-result rows


In [6]:
# Required by the brief: show .shape, .head(), .info()

print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"Rows × columns: {df.shape}")

print("\n" + "=" * 60)
print("HEAD (first 5 rows)")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("INFO (column types and non-null counts)")
print("=" * 60)
df.info()

SHAPE
Rows × columns: (2618, 19)

HEAD (first 5 rows)


,season,round,race_name,circuit_id,circuit_name,country,date,position,position_text,points,grid,laps,status,driver_id,driver_code,driver_first,driver_last,constructor_id,constructor_name
0,2020,1,Austrian Grand Prix,red_bull_ring,Red Bull Ring,Austria,2020-07-05,1,1,25.0,1,71,Finished,bottas,BOT,Valtteri,Bottas,mercedes,Mercedes
1,2020,1,Austrian Grand Prix,red_bull_ring,Red Bull Ring,Austria,2020-07-05,2,2,18.0,7,71,Finished,leclerc,LEC,Charles,Leclerc,ferrari,Ferrari
2,2020,1,Austrian Grand Prix,red_bull_ring,Red Bull Ring,Austria,2020-07-05,3,3,16.0,3,71,Finished,norris,NOR,Lando,Norris,mclaren,McLaren
3,2020,1,Austrian Grand Prix,red_bull_ring,Red Bull Ring,Austria,2020-07-05,4,4,12.0,5,71,Finished,hamilton,HAM,Lewis,Hamilton,mercedes,Mercedes
4,2020,1,Austrian Grand Prix,red_bull_ring,Red Bull Ring,Austria,2020-07-05,5,5,10.0,8,71,Finished,sainz,SAI,Carlos,Sainz,mclaren,McLaren



INFO (column types and non-null counts)
<class 'pandas.DataFrame'>
RangeIndex: 2618 entries, 0 to 2617
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   season            2618 non-null   int64  
 1   round             2618 non-null   int64  
 2   race_name         2618 non-null   str    
 3   circuit_id        2618 non-null   str    
 4   circuit_name      2618 non-null   str    
 5   country           2618 non-null   str    
 6   date              2618 non-null   str    
 7   position          2618 non-null   int64  
 8   position_text     2618 non-null   str    
 9   points            2618 non-null   float64
 10  grid              2618 non-null   int64  
 11  laps              2618 non-null   int64  
 12  status            2618 non-null   str    
 13  driver_id         2618 non-null   str    
 14  driver_code       2618 non-null   str    
 15  driver_first      2618 non-null   str    
 16  driver_last 

In [7]:
# Save two files to data/raw/:
# 1. The raw API JSON (the original nested structure) — immutable checkpoint
# 2. The flat DataFrame as CSV — convenient for re-loading in later notebooks

raw_json_path = DATA_RAW / "f1_results_2020_2025.json"
raw_csv_path  = DATA_RAW / "f1_results_2020_2025.csv"

# Save the raw nested JSON exactly as we received it from the API
raw_json_path.write_text(json.dumps(all_races, indent=2))

# Save the flat DataFrame as CSV
df.to_csv(raw_csv_path, index=False)

# Verify both files exist and report their sizes
for path in [raw_json_path, raw_csv_path]:
    size_kb = path.stat().st_size / 1024
    print(f"✅ Saved {path.name}: {size_kb:,.1f} KB")

✅ Saved f1_results_2020_2025.json: 2,660.1 KB
✅ Saved f1_results_2020_2025.csv: 379.1 KB
